In [1]:
import sys
import os

api_path = os.path.abspath(os.path.join("..", "scripts", "api"))

if api_path not in sys.path:
    sys.path.append(api_path)
    

In [2]:
import pandas as pd
from datetime import datetime


from dotenv import load_dotenv

from api.fetch_historical_match_events import fetch_match_events_for_date, extract_event_metadata
from api.fetch_historical_ags_odds import fetch_ags_odds_for_event

/Users/tylermartins/Documents/PitchSights/ps-betting/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
load_dotenv()
API_KEY = os.getenv("ODDS_API_KEY")

In [4]:
# INPUTS
season = '2024-2025'
league = 'Premier-League'

api_league = 'soccer_epl'

In [10]:
from datetime import datetime

def convert_to_iso(date_str):
    """Convert 'Friday August 16, 2024' to '2024-08-16T00:00:00Z'"""
    dt = datetime.strptime(date_str, "%A %B %d, %Y")
    return dt.strftime("%Y-%m-%dT00:00:00Z")

def normalize_commence_time(commence_time_iso):
    """Extract 'YYYY-MM-DD' from 'YYYY-MM-DDTHH:MM:SSZ'"""
    return datetime.fromisoformat(commence_time_iso.replace("Z", "")).date().isoformat()


date_str = "Friday August 16, 2024"
iso_date = convert_to_iso(date_str)
iso_date

'2024-08-16T00:00:00Z'

In [8]:
# --- Step 1: Load match dates from CSV and fetch event metadata ---

# Step 1.1 — Load the CSV
csv_path = f"../data/raw/ags_match_stats_{league}_{season}.csv"
df = pd.read_csv(csv_path)

# Step 1.2 — Extract unique match dates
raw_dates = df['match_date'].dropna().unique()

# Step 1.3 — Fetch match events for each date and deduplicate by event_id
event_meta_map = {}  # event_id → metadata (commence_time, home_team, away_team)

for raw_date in raw_dates:
    try:
        iso_date = convert_to_iso(raw_date)
        print(f"\n📅 Fetching match events for: {raw_date} → {iso_date}")

        response = fetch_match_events_for_date(api_league, iso_date)
        events = extract_event_metadata(response)

        if not events:
            print("  ⚠️ No events found.")
            continue

        for event in events:
            event_id = event["event_id"]
            if event_id not in event_meta_map:
                event_meta_map[event_id] = {
                    "commence_time": event["commence_time"],
                    "home_team": event["home_team"],
                    "away_team": event["away_team"]
                }

        print(f"✅ Added {len(events)} events from this date (after deduplication).")

    except Exception as e:
        print(f"❌ Error processing {raw_date}: {e}")



📅 Fetching match events for: Friday August 16, 2024 → 2024-08-16T00:00:00Z
✅ Added 10 events from this date (after deduplication).

📅 Fetching match events for: Saturday August 17, 2024 → 2024-08-17T00:00:00Z
✅ Added 9 events from this date (after deduplication).

📅 Fetching match events for: Sunday August 18, 2024 → 2024-08-18T00:00:00Z
✅ Added 13 events from this date (after deduplication).

📅 Fetching match events for: Monday August 19, 2024 → 2024-08-19T00:00:00Z
✅ Added 20 events from this date (after deduplication).

📅 Fetching match events for: Saturday August 24, 2024 → 2024-08-24T00:00:00Z
✅ Added 20 events from this date (after deduplication).

📅 Fetching match events for: Sunday August 25, 2024 → 2024-08-25T00:00:00Z
✅ Added 13 events from this date (after deduplication).

📅 Fetching match events for: Saturday August 31, 2024 → 2024-08-31T00:00:00Z
✅ Added 20 events from this date (after deduplication).

📅 Fetching match events for: Sunday September 1, 2024 → 2024-09-01T00:

In [9]:
event_meta_map


{'7579d22151b7be3f590e9e338de9dff6': {'commence_time': '2024-08-16T19:00:00Z',
  'home_team': 'Manchester United',
  'away_team': 'Fulham'},
 '295b86b1f5f7bc00e459da5b998cba57': {'commence_time': '2024-08-17T11:30:00Z',
  'home_team': 'Ipswich Town',
  'away_team': 'Liverpool'},
 '9d2b152101cb8d09b84f6c58728d9e41': {'commence_time': '2024-08-17T14:00:00Z',
  'home_team': 'Arsenal',
  'away_team': 'Wolverhampton Wanderers'},
 'e09a13134dbd8919ccc4805a59984ddc': {'commence_time': '2024-08-17T14:00:00Z',
  'home_team': 'Nottingham Forest',
  'away_team': 'Bournemouth'},
 '1ded735e9bdcd609e5bd48cb1ba05c8e': {'commence_time': '2024-08-17T14:00:00Z',
  'home_team': 'Everton',
  'away_team': 'Brighton and Hove Albion'},
 '9520b4cd30c7dd2f25346dea1e0cb685': {'commence_time': '2024-08-17T14:00:00Z',
  'home_team': 'Newcastle United',
  'away_team': 'Southampton'},
 '846ea0dcf3f8e6e3259bafdbd9de736c': {'commence_time': '2024-08-17T16:30:00Z',
  'home_team': 'West Ham United',
  'away_team': 'Ast

In [11]:
# --- Step 2: Fetch SOT odds for each unique match event ---

all_odds_rows = []

for event_id, event in event_meta_map.items():
    home_team = event.get("home_team")
    away_team = event.get("away_team")
    commence_time = event.get("commence_time")
    match_date = normalize_commence_time(commence_time)

    print(f"\n🎯 {home_team} vs {away_team} | {match_date} | Event ID: {event_id}")

    try:
        odds_response = fetch_ags_odds_for_event(api_league, event_id, commence_time)
        bookmakers = odds_response.get("data", {}).get("bookmakers", [])

        if not bookmakers:
            print("  ❌ No SOT odds found.")
            continue

        for bookmaker in bookmakers:
            bookmaker_name = bookmaker.get("title")
            for market in bookmaker.get("markets", []):
                if market.get("key") != "player_goal_scorer_anytime":
                    continue
                for outcome in market.get("outcomes", []):
                    all_odds_rows.append({
                        "match_date": match_date,
                        "event_id": event_id,
                        "home_team": home_team,
                        "away_team": away_team,
                        "bookmaker": bookmaker_name,
                        "player": outcome.get("description"),
                        "line": outcome.get("point"),
                        "direction": outcome.get("name"),
                        "odds": outcome.get("price")
                    })

    except Exception as e:
        print(f"  ⚠️ Error fetching odds for event {event_id}: {e}")



🎯 Manchester United vs Fulham | 2024-08-16 | Event ID: 7579d22151b7be3f590e9e338de9dff6

🎯 Ipswich Town vs Liverpool | 2024-08-17 | Event ID: 295b86b1f5f7bc00e459da5b998cba57

🎯 Arsenal vs Wolverhampton Wanderers | 2024-08-17 | Event ID: 9d2b152101cb8d09b84f6c58728d9e41

🎯 Nottingham Forest vs Bournemouth | 2024-08-17 | Event ID: e09a13134dbd8919ccc4805a59984ddc

🎯 Everton vs Brighton and Hove Albion | 2024-08-17 | Event ID: 1ded735e9bdcd609e5bd48cb1ba05c8e

🎯 Newcastle United vs Southampton | 2024-08-17 | Event ID: 9520b4cd30c7dd2f25346dea1e0cb685

🎯 West Ham United vs Aston Villa | 2024-08-17 | Event ID: 846ea0dcf3f8e6e3259bafdbd9de736c

🎯 Brentford vs Crystal Palace | 2024-08-18 | Event ID: 291eafbf7314c7e76ef9c4d5fcfc51f7

🎯 Chelsea vs Manchester City | 2024-08-18 | Event ID: fa18efa02f086a86224b6c5bfc582d70

🎯 Leicester City vs Tottenham Hotspur | 2024-08-19 | Event ID: 36808dc52374b40df08cab6b1d5664d1

🎯 Brighton and Hove Albion vs Manchester United | 2024-08-24 | Event ID: 9b87

In [12]:
odds_df = pd.DataFrame(all_odds_rows)
odds_df


,match_date,event_id,home_team,away_team,bookmaker,player,line,direction,odds
0,2024-08-16,7579d22151b7be3f590e9e338de9dff6,Manchester United,Fulham,Bovada,Harry Amass,None,Yes,13.0
1,2024-08-16,7579d22151b7be3f590e9e338de9dff6,Manchester United,Fulham,Bovada,Jonny Evans,None,Yes,13.0
2,2024-08-16,7579d22151b7be3f590e9e338de9dff6,Manchester United,Fulham,Bovada,Tom Cairney,None,Yes,11.0
3,2024-08-16,7579d22151b7be3f590e9e338de9dff6,Manchester United,Fulham,Bovada,Christian Eriksen,None,Yes,6.5
4,2024-08-16,7579d22151b7be3f590e9e338de9dff6,Manchester United,Fulham,Bovada,Casemiro,None,Yes,5.6
...,...,...,...,...,...,...,...,...,...
41219,2025-05-25,f7ee35bd5614661f0c83e59885c2a933,Ipswich Town,West Ham United,BetMGM,Axel Tuanzebe,None,Yes,18.0
41220,2025-05-25,f7ee35bd5614661f0c83e59885c2a933,Ipswich Town,West Ham United,BetMGM,Vladimír Coufal,None,Yes,18.0
41221,2025-05-25,f7ee35bd5614661f0c83e59885c2a933,Ipswich Town,West Ham United,BetMGM,Max Kilman,None,Yes,21.0
41222,2025-05-25,f7ee35bd5614661f0c83e59885c2a933,Ipswich Town,West Ham United,BetMGM,Jean-Clair Todibo,None,Yes,21.0


In [13]:
output_path = f"../data/raw/historical_ags_odds_output_{league}_{season}.csv"
odds_df.to_csv(output_path, index=False)
print(f"\n✅ Saved {len(odds_df)} rows to {output_path}")


✅ Saved 41224 rows to ../data/raw/historical_ags_odds_output_Premier-League_2024-2025.csv
